# EDA Case Study: Employee Attrition & Performance Analysis

## HR Analytics Deep Dive

---

## Business Context

**TechFlow Inc.** is a mid-sized technology company experiencing higher-than-expected employee turnover. The HR department has collected data on current and former employees to understand the factors driving attrition.

### Why Employee Attrition Matters

| Impact Area | Cost/Effect |
|-------------|-------------|
| **Recruitment** | $4,000 - $7,000 per hire |
| **Training** | 3-6 months to full productivity |
| **Knowledge Loss** | Institutional knowledge walks out the door |
| **Team Morale** | Remaining employees may feel overworked |
| **Customer Impact** | Service disruptions during transitions |

**Industry benchmark**: Average tech industry attrition is ~13-15%. TechFlow's rate seems higher.

### Stakeholders

- **HR Leadership**: Need to reduce turnover and improve retention strategies
- **Department Managers**: Want to identify at-risk team members early
- **Executive Team**: Concerned about costs and competitive talent market
- **Finance**: Need to budget for recruitment and training

### Key Business Questions

1. **What is our actual attrition rate?** How does it compare to industry benchmarks?
2. **Which departments/roles have the highest turnover?** Where should we focus retention efforts?
3. **What employee characteristics predict attrition?** Age, tenure, satisfaction, compensation?
4. **Are there early warning signs?** Can we identify at-risk employees before they leave?
5. **What actionable recommendations can we make?** How can HR intervene effectively?

---

## Dataset Overview

The HR team has provided employee data with the following information:

| Column | Description |
|--------|-------------|
| employee_id | Unique identifier |
| age | Employee age |
| gender | Gender identity |
| department | Work department |
| job_level | Career level |
| years_at_company | Tenure |
| monthly_income | Salary |
| performance_rating | Annual review score (1-5) |
| work_life_balance | Self-reported (1-5) |
| job_satisfaction | Self-reported (1-5) |
| overtime | Works overtime? |
| distance_from_home | Commute distance |
| training_hours_last_year | Development hours |
| num_companies_worked | Prior employers |
| attrition | Left the company? |

**Data Quality Warning**: This dataset has NOT been cleaned. Expect messy data!

---

# EDA case study: employee attrition — Assessment (D007)

**Time:** 60 minutes · **Marks:** 100 · **Format:** coding notebook plus written justification, marked by a human.

**Data:** `../../_assets/data/hr_employee_data.csv`

**Rules:** only what has been taught up to this day; comment the *why*, not the *what*; a printed number without its justification earns half the marks.

Steps 1 to 5 above the *Student Exercises* heading are the worked walkthrough: run them so the cleaned data exists, but they are not marked. Only Exercises 1 to 5 are.

| Task | Bloom level | Marks | What is marked |
|---|---|---|---|
| Exercise 1: basic inspection | understand | 10 | Employees per department with a labelled bar chart; mean tenure per department; the longest named. |
| Exercise 2: attrition by job level | apply | 20 | Attrition RATE per job level (a mean of a 0/1 column or a normalised count), highest and lowest named, one chart. |
| Exercise 3: compensation brackets | apply | 20 | pd.cut brackets with the stated edges; attrition rate per bracket; a one-line answer to 'is there a pattern'. |
| Exercise 4: satisfaction | analyse | 20 | Means for leavers versus stayers on both scores; side-by-side box plots; the low-satisfaction subgroup's attrition; conclusions that cite the numbers. |
| Exercise 5: at-risk profile | evaluate | 30 | Three criteria justified from Exercises 2 to 4; an is_at_risk column; the share flagged and the attrition rate inside the flag versus outside; a written profile. |

Anything headed **Bonus (not marked)** is optional stretch material.

## Setup

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


---

# Step 1: LOAD the Data

In [ ]:
# Load the dataset
df = pd.read_csv('../../00-data/hr_employee_data.csv')

print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

---

# Step 2: INSPECT the Data

Let's explore the data to understand its structure and identify quality issues.

In [ ]:
# First look at the data
df.head(10)

In [ ]:
# Data types
print("Data Types:")
print("=" * 40)
print(df.dtypes)

In [ ]:
# Dataset info
print("Dataset Info:")
print("=" * 40)
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Check missing values
print("Missing Values:")
print("=" * 40)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Count'] > 0])

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates} ({duplicates/len(df)*100:.2f}%)")

## 2.1 Investigate Categorical Data Issues

Let's look at the unique values in our categorical columns to identify inconsistencies.

In [ ]:
# Check unique values in categorical columns
categorical_cols = ['gender', 'department', 'job_level', 'overtime', 'attrition']

print("Unique Values in Categorical Columns:")
print("=" * 60)

for col in categorical_cols:
    unique_vals = df[col].unique()
    print(f"\n{col} ({len(unique_vals)} unique values):")
    print(f"  {list(unique_vals)}")

In [ ]:
# Detailed value counts for each categorical column
print("\nValue Counts (showing the mess):")
print("=" * 60)

for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())

## 2.2 Investigate Numerical Data Issues

Let's check for impossible values and outliers in numerical columns.

In [ ]:
# Check numerical columns for anomalies
numerical_cols = ['age', 'years_at_company', 'monthly_income', 'distance_from_home']

print("Numerical Column Ranges (look for impossible values):")
print("=" * 60)

for col in numerical_cols:
    print(f"\n{col}:")
    print(f"  Min: {df[col].min()}")
    print(f"  Max: {df[col].max()}")
    print(f"  Mean: {df[col].mean():.2f}")
    print(f"  Negative values: {(df[col] < 0).sum()}")

In [ ]:
# Check for logical errors: years_at_company > age - 18 (impossible)
df['tenure_check'] = df['years_at_company'] > (df['age'] - 18)
logical_errors = df['tenure_check'].sum()
print(f"Logical errors (tenure > possible working years): {logical_errors}")

if logical_errors > 0:
    print("\nSample of logical errors:")
    display(df[df['tenure_check']][['employee_id', 'age', 'years_at_company']].head())

# Clean up temp column
df.drop('tenure_check', axis=1, inplace=True)

### Data Quality Summary

Based on our inspection, we found the following issues:

**Categorical Issues:**
- `gender`: Multiple formats (Male/male/M/m, Female/female/F/f, etc.)
- `department`: Typos, case issues, extra whitespace
- `job_level`: Typos, case issues, extra whitespace
- `overtime`: Inconsistent values (Yes/yes/Y/1, No/no/N/0)
- `attrition`: Inconsistent case

**Numerical Issues:**
- `age`: Impossible values (negative, >100, <18)
- `monthly_income`: Negative values, extreme outliers
- `distance_from_home`: Negative values, extreme outliers
- `years_at_company`: Logical errors (tenure > possible working years)
- `performance_rating`: ~7% missing
- `job_satisfaction`: ~5% missing

**Other Issues:**
- Duplicate rows (~1%)

---

# Step 3: CLEAN the Data

Now we'll systematically clean each issue we identified.

## 3.1 Clean Categorical Variables

In [ ]:
# Create a clean copy
df_clean = df.copy()

# Clean GENDER
print("Cleaning 'gender'...")
print(f"  Before: {df_clean['gender'].nunique()} unique values")

# Standardize gender
gender_mapping = {
    'male': 'Male', 'MALE': 'Male', 'M': 'Male', 'm': 'Male',
    'female': 'Female', 'FEMALE': 'Female', 'F': 'Female', 'f': 'Female',
    'other': 'Other', 'OTHER': 'Other'
}
df_clean['gender'] = df_clean['gender'].replace(gender_mapping)

print(f"  After: {df_clean['gender'].nunique()} unique values")
print(f"  Values: {df_clean['gender'].unique()}")

In [ ]:
# Clean DEPARTMENT
print("Cleaning 'department'...")
print(f"  Before: {df_clean['department'].nunique()} unique values")

# Step 1: Strip whitespace and convert to title case
df_clean['department'] = df_clean['department'].str.strip().str.title()

# Step 2: Fix typos and standardize names
dept_mapping = {
    'Saless': 'Sales',
    'Enginering': 'Engineering',
    'Human Resources': 'HR',
    'Marketng': 'Marketing',
    'Finace': 'Finance',
    'Operatons': 'Operations'
}
df_clean['department'] = df_clean['department'].replace(dept_mapping)

print(f"  After: {df_clean['department'].nunique()} unique values")
print(f"  Values: {df_clean['department'].unique()}")

In [ ]:
# Clean JOB_LEVEL
print("Cleaning 'job_level'...")
print(f"  Before: {df_clean['job_level'].nunique()} unique values")

# Step 1: Strip whitespace and convert to title case
df_clean['job_level'] = df_clean['job_level'].str.strip().str.title()

# Step 2: Fix typos
level_mapping = {
    'Enrty': 'Entry',
    'Midd': 'Mid',
    'Senoir': 'Senior',
    'Manger': 'Manager',
    'Diretor': 'Director'
}
df_clean['job_level'] = df_clean['job_level'].replace(level_mapping)

print(f"  After: {df_clean['job_level'].nunique()} unique values")
print(f"  Values: {df_clean['job_level'].unique()}")

In [ ]:
# Clean OVERTIME
print("Cleaning 'overtime'...")
print(f"  Before: {df_clean['overtime'].nunique()} unique values")

# Map all variations to Yes/No
overtime_mapping = {
    'yes': 'Yes', 'YES': 'Yes', 'Y': 'Yes', 'y': 'Yes', '1': 'Yes',
    'no': 'No', 'NO': 'No', 'N': 'No', 'n': 'No', '0': 'No'
}
df_clean['overtime'] = df_clean['overtime'].replace(overtime_mapping)

print(f"  After: {df_clean['overtime'].nunique()} unique values")
print(f"  Values: {df_clean['overtime'].unique()}")

In [ ]:
# Clean ATTRITION
print("Cleaning 'attrition'...")
print(f"  Before: {df_clean['attrition'].nunique()} unique values")

# Standardize to Yes/No
attrition_mapping = {
    'yes': 'Yes', 'YES': 'Yes',
    'no': 'No', 'NO': 'No'
}
df_clean['attrition'] = df_clean['attrition'].replace(attrition_mapping)

print(f"  After: {df_clean['attrition'].nunique()} unique values")
print(f"  Values: {df_clean['attrition'].unique()}")

## 3.2 Clean Numerical Variables

In [ ]:
# Clean AGE - remove impossible values
print("Cleaning 'age'...")
print(f"  Invalid ages (< 18 or > 70): {((df_clean['age'] < 18) | (df_clean['age'] > 70)).sum()}")

# Replace impossible ages with NaN, then impute with median
df_clean.loc[(df_clean['age'] < 18) | (df_clean['age'] > 70), 'age'] = np.nan
age_median = df_clean['age'].median()
df_clean['age'].fillna(age_median, inplace=True)

print(f"  Imputed with median: {age_median}")
print(f"  New range: {df_clean['age'].min()} - {df_clean['age'].max()}")

In [ ]:
# Clean MONTHLY_INCOME - handle negative and extreme values
print("Cleaning 'monthly_income'...")
print(f"  Negative values: {(df_clean['monthly_income'] < 0).sum()}")
print(f"  Extreme high (> 50000): {(df_clean['monthly_income'] > 50000).sum()}")

# Replace invalid values with NaN
df_clean.loc[df_clean['monthly_income'] < 0, 'monthly_income'] = np.nan
df_clean.loc[df_clean['monthly_income'] < 1000, 'monthly_income'] = np.nan  # Unrealistically low

# Cap extreme high values at 99th percentile
income_cap = df_clean['monthly_income'].quantile(0.99)
df_clean.loc[df_clean['monthly_income'] > income_cap, 'monthly_income'] = income_cap

# Impute missing with median
income_median = df_clean['monthly_income'].median()
df_clean['monthly_income'].fillna(income_median, inplace=True)

print(f"  Capped at: ${income_cap:.2f}")
print(f"  Imputed with median: ${income_median:.2f}")
print(f"  New range: ${df_clean['monthly_income'].min():.2f} - ${df_clean['monthly_income'].max():.2f}")

In [ ]:
# Clean DISTANCE_FROM_HOME - handle negative and extreme values
print("Cleaning 'distance_from_home'...")
print(f"  Negative values: {(df_clean['distance_from_home'] < 0).sum()}")
print(f"  Extreme (> 100 miles): {(df_clean['distance_from_home'] > 100).sum()}")

# Replace invalid values with NaN
df_clean.loc[df_clean['distance_from_home'] < 0, 'distance_from_home'] = np.nan
df_clean.loc[df_clean['distance_from_home'] > 100, 'distance_from_home'] = np.nan

# Impute with median
dist_median = df_clean['distance_from_home'].median()
df_clean['distance_from_home'].fillna(dist_median, inplace=True)

print(f"  Imputed with median: {dist_median:.1f} miles")
print(f"  New range: {df_clean['distance_from_home'].min():.1f} - {df_clean['distance_from_home'].max():.1f} miles")

In [ ]:
# Clean YEARS_AT_COMPANY - fix logical errors
print("Cleaning 'years_at_company'...")

# Cap tenure at (age - 18) to fix logical errors
max_possible_tenure = df_clean['age'] - 18
df_clean['years_at_company'] = df_clean[['years_at_company', 'age']].apply(
    lambda x: min(x['years_at_company'], x['age'] - 18) if x['years_at_company'] > (x['age'] - 18) else x['years_at_company'],
    axis=1
)

# Ensure non-negative
df_clean['years_at_company'] = df_clean['years_at_company'].clip(lower=0)

print(f"  New range: {df_clean['years_at_company'].min():.1f} - {df_clean['years_at_company'].max():.1f} years")

In [ ]:
# Handle missing PERFORMANCE_RATING and JOB_SATISFACTION
print("Handling missing values in survey data...")

# Performance rating - impute with mode (most common rating)
perf_mode = df_clean['performance_rating'].mode()[0]
df_clean['performance_rating'].fillna(perf_mode, inplace=True)
print(f"  performance_rating: imputed {df['performance_rating'].isnull().sum()} missing with mode ({perf_mode})")

# Job satisfaction - impute with median
sat_median = df_clean['job_satisfaction'].median()
df_clean['job_satisfaction'].fillna(sat_median, inplace=True)
print(f"  job_satisfaction: imputed {df['job_satisfaction'].isnull().sum()} missing with median ({sat_median})")

## 3.3 Remove Duplicates

In [ ]:
# Remove duplicate rows
print(f"Before removing duplicates: {len(df_clean)} rows")

df_clean = df_clean.drop_duplicates()

print(f"After removing duplicates: {len(df_clean)} rows")
print(f"Removed: {len(df) - len(df_clean)} duplicate rows")

## 3.4 Verify Cleaning Results

In [ ]:
# Final verification
print("=" * 60)
print("DATA CLEANING VERIFICATION")
print("=" * 60)

print(f"\nDataset shape: {df_clean.shape}")
print(f"\nMissing values: {df_clean.isnull().sum().sum()}")
print(f"Duplicates: {df_clean.duplicated().sum()}")

print("\nCategorical columns - unique values:")
for col in ['gender', 'department', 'job_level', 'overtime', 'attrition']:
    print(f"  {col}: {df_clean[col].unique()}")

print("\nNumerical columns - ranges:")
for col in ['age', 'monthly_income', 'distance_from_home', 'years_at_company']:
    print(f"  {col}: {df_clean[col].min():.1f} to {df_clean[col].max():.1f}")

---

# Step 4: EXPLORE the Data

Now that our data is clean, let's answer the business questions through exploration.

## 4.1 Attrition Overview

**Business Question**: What is our actual attrition rate?

In [ ]:
# Calculate overall attrition rate
attrition_counts = df_clean['attrition'].value_counts()
attrition_rate = (attrition_counts['Yes'] / len(df_clean)) * 100

print("ATTRITION OVERVIEW")
print("=" * 40)
print(f"\nTotal employees: {len(df_clean)}")
print(f"Employees who left: {attrition_counts['Yes']}")
print(f"Employees who stayed: {attrition_counts['No']}")
print(f"\nAttrition Rate: {attrition_rate:.1f}%")
print(f"Industry benchmark: 13-15%")
print(f"Status: {'ABOVE BENCHMARK - Action needed!' if attrition_rate > 15 else 'Within acceptable range'}")

In [ ]:
# Visualize attrition
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(attrition_counts, labels=['Stayed', 'Left'], autopct='%1.1f%%', 
            colors=colors, explode=[0, 0.05], shadow=True, startangle=90)
axes[0].set_title('Employee Attrition Distribution', fontsize=14)

# Bar chart
axes[1].bar(['Stayed', 'Left'], attrition_counts.values, color=colors, edgecolor='black')
axes[1].set_ylabel('Number of Employees')
axes[1].set_title('Attrition Count', fontsize=14)
for i, v in enumerate(attrition_counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4.2 Department Analysis

**Business Question**: Which departments have the highest attrition?

In [ ]:
# Attrition by department
dept_attrition = df_clean.groupby('department').agg({
    'employee_id': 'count',
    'attrition': lambda x: (x == 'Yes').sum()
}).rename(columns={'employee_id': 'Total', 'attrition': 'Left'})

dept_attrition['Attrition_Rate'] = (dept_attrition['Left'] / dept_attrition['Total'] * 100).round(1)
dept_attrition = dept_attrition.sort_values('Attrition_Rate', ascending=False)

print("ATTRITION BY DEPARTMENT")
print("=" * 50)
print(dept_attrition)

In [ ]:
# Visualize department attrition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Department sizes
dept_sizes = df_clean['department'].value_counts()
axes[0].bar(dept_sizes.index, dept_sizes.values, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Department')
axes[0].set_ylabel('Number of Employees')
axes[0].set_title('Department Sizes')
axes[0].tick_params(axis='x', rotation=45)

# Attrition rates by department
colors = ['#e74c3c' if rate > 20 else '#f39c12' if rate > 15 else '#2ecc71' 
          for rate in dept_attrition['Attrition_Rate']]
axes[1].barh(dept_attrition.index, dept_attrition['Attrition_Rate'], color=colors, edgecolor='black')
axes[1].axvline(x=15, color='red', linestyle='--', label='Industry benchmark (15%)')
axes[1].set_xlabel('Attrition Rate (%)')
axes[1].set_title('Attrition Rate by Department')
axes[1].legend()

# Add percentage labels
for i, (idx, row) in enumerate(dept_attrition.iterrows()):
    axes[1].text(row['Attrition_Rate'] + 0.5, i, f"{row['Attrition_Rate']}%", va='center')

plt.tight_layout()
plt.show()

## 4.3 Job Level Analysis

**Business Question**: Which job levels have the highest turnover?

In [ ]:
# Attrition by job level
level_order = ['Entry', 'Mid', 'Senior', 'Manager', 'Director']

level_attrition = df_clean.groupby('job_level').agg({
    'employee_id': 'count',
    'attrition': lambda x: (x == 'Yes').sum()
}).rename(columns={'employee_id': 'Total', 'attrition': 'Left'})

level_attrition['Attrition_Rate'] = (level_attrition['Left'] / level_attrition['Total'] * 100).round(1)
level_attrition = level_attrition.reindex(level_order)

print("ATTRITION BY JOB LEVEL")
print("=" * 50)
print(level_attrition)

In [ ]:
# Visualize job level attrition
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(level_order))
width = 0.35

bars1 = ax.bar(x - width/2, level_attrition['Total'], width, label='Total Employees', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, level_attrition['Left'], width, label='Left', color='coral', edgecolor='black')

ax.set_xlabel('Job Level')
ax.set_ylabel('Count')
ax.set_title('Employee Count and Attrition by Job Level')
ax.set_xticks(x)
ax.set_xticklabels(level_order)
ax.legend()

# Add attrition rate labels
for i, (level, row) in enumerate(level_attrition.iterrows()):
    ax.annotate(f'{row["Attrition_Rate"]}%', 
                xy=(i + width/2, row['Left']), 
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontweight='bold', color='red')

plt.tight_layout()
plt.show()

## 4.4 Compensation Analysis

**Business Question**: Is there a relationship between income and attrition?

In [ ]:
# Create income brackets
df_clean['income_bracket'] = pd.cut(df_clean['monthly_income'], 
                                     bins=[0, 4000, 7000, 12000, float('inf')],
                                     labels=['Low (<$4K)', 'Medium ($4-7K)', 'High ($7-12K)', 'Very High (>$12K)'])

# Attrition by income bracket
income_attrition = df_clean.groupby('income_bracket').agg({
    'employee_id': 'count',
    'attrition': lambda x: (x == 'Yes').sum()
}).rename(columns={'employee_id': 'Total', 'attrition': 'Left'})

income_attrition['Attrition_Rate'] = (income_attrition['Left'] / income_attrition['Total'] * 100).round(1)

print("ATTRITION BY INCOME BRACKET")
print("=" * 50)
print(income_attrition)

In [ ]:
# Visualize income and attrition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot of income by attrition status
sns.boxplot(data=df_clean, x='attrition', y='monthly_income', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_xlabel('Attrition')
axes[0].set_ylabel('Monthly Income ($)')
axes[0].set_title('Income Distribution: Stayed vs Left')

# Attrition rate by income bracket
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#27ae60']
axes[1].bar(income_attrition.index.astype(str), income_attrition['Attrition_Rate'], 
            color=colors, edgecolor='black')
axes[1].axhline(y=15, color='red', linestyle='--', label='Benchmark (15%)')
axes[1].set_xlabel('Income Bracket')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_title('Attrition Rate by Income Level')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# Statistical comparison
left_income = df_clean[df_clean['attrition'] == 'Yes']['monthly_income'].mean()
stayed_income = df_clean[df_clean['attrition'] == 'No']['monthly_income'].mean()
print(f"\nAverage income of employees who LEFT: ${left_income:,.2f}")
print(f"Average income of employees who STAYED: ${stayed_income:,.2f}")
print(f"Difference: ${stayed_income - left_income:,.2f} ({(stayed_income - left_income)/left_income*100:.1f}% higher)")

## 4.5 Satisfaction Analysis

**Business Question**: How do satisfaction metrics relate to attrition?

In [ ]:
# Compare satisfaction metrics between those who left and stayed
satisfaction_comparison = df_clean.groupby('attrition')[['job_satisfaction', 'work_life_balance', 'performance_rating']].mean()

print("SATISFACTION METRICS BY ATTRITION STATUS")
print("=" * 50)
print(satisfaction_comparison.round(2))

In [ ]:
# Visualize satisfaction metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['job_satisfaction', 'work_life_balance', 'performance_rating']
titles = ['Job Satisfaction', 'Work-Life Balance', 'Performance Rating']

for ax, metric, title in zip(axes, metrics, titles):
    sns.boxplot(data=df_clean, x='attrition', y=metric, ax=ax, palette=['#2ecc71', '#e74c3c'])
    ax.set_xlabel('Attrition')
    ax.set_ylabel('Score (1-5)')
    ax.set_title(title)
    ax.set_ylim([0.5, 5.5])

plt.suptitle('Satisfaction Metrics: Employees Who Stayed vs Left', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Attrition rate by satisfaction level
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Job satisfaction
sat_attrition = df_clean.groupby('job_satisfaction').apply(
    lambda x: (x['attrition'] == 'Yes').sum() / len(x) * 100
)
axes[0].bar(sat_attrition.index, sat_attrition.values, color='coral', edgecolor='black')
axes[0].set_xlabel('Job Satisfaction Score')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].set_title('Attrition Rate by Job Satisfaction')
axes[0].axhline(y=15, color='red', linestyle='--', label='Benchmark')
axes[0].legend()

# Work-life balance
wlb_attrition = df_clean.groupby('work_life_balance').apply(
    lambda x: (x['attrition'] == 'Yes').sum() / len(x) * 100
)
axes[1].bar(wlb_attrition.index, wlb_attrition.values, color='teal', edgecolor='black')
axes[1].set_xlabel('Work-Life Balance Score')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_title('Attrition Rate by Work-Life Balance')
axes[1].axhline(y=15, color='red', linestyle='--', label='Benchmark')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4.6 Correlation Analysis

**Business Question**: What factors are most correlated with attrition?

In [ ]:
# Create numeric version of attrition for correlation
df_clean['attrition_numeric'] = (df_clean['attrition'] == 'Yes').astype(int)

# Calculate correlations with attrition
numeric_cols = ['age', 'years_at_company', 'monthly_income', 'performance_rating',
                'work_life_balance', 'job_satisfaction', 'distance_from_home',
                'training_hours_last_year', 'num_companies_worked', 'attrition_numeric']

correlation_matrix = df_clean[numeric_cols].corr()

# Get correlations with attrition
attrition_corr = correlation_matrix['attrition_numeric'].drop('attrition_numeric').sort_values()

print("CORRELATION WITH ATTRITION")
print("=" * 50)
print("(Positive = higher value associated with leaving)")
print("(Negative = higher value associated with staying)")
print()
for var, corr in attrition_corr.items():
    direction = "↑" if corr > 0 else "↓"
    print(f"{var:30} {direction} {corr:+.3f}")

In [ ]:
# Visualize correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Heatmap: Employee Metrics', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Horizontal bar chart of correlations with attrition
plt.figure(figsize=(10, 6))

colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in attrition_corr.values]
plt.barh(attrition_corr.index, attrition_corr.values, color=colors, edgecolor='black')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.xlabel('Correlation with Attrition')
plt.title('Factors Correlated with Employee Attrition\n(Red = increases attrition, Green = decreases attrition)')
plt.tight_layout()
plt.show()

---

# Step 5: SUMMARIZE - Key Findings

In [ ]:
print("\n" + "#"*70)
print("#" + " "*18 + "EXECUTIVE SUMMARY" + " "*31 + "#")
print("#" + " "*15 + "TechFlow Inc. - HR Analytics" + " "*22 + "#")
print("#"*70)

print(f"""
OVERVIEW:
=========
Total employees analyzed: {len(df_clean):,}
Overall attrition rate: {attrition_rate:.1f}%
Industry benchmark: 13-15%
Status: {'ABOVE BENCHMARK - Action Required' if attrition_rate > 15 else 'Within acceptable range'}

KEY FINDINGS:
=============

1. DEPARTMENT HOTSPOTS:
   - Highest attrition: {dept_attrition['Attrition_Rate'].idxmax()} ({dept_attrition['Attrition_Rate'].max():.1f}%)
   - Lowest attrition: {dept_attrition['Attrition_Rate'].idxmin()} ({dept_attrition['Attrition_Rate'].min():.1f}%)
   - Recommendation: Focus retention efforts on high-attrition departments

2. JOB LEVEL PATTERNS:
   - Entry-level employees have highest turnover
   - Senior/Director levels show better retention
   - Recommendation: Improve onboarding and career progression for entry-level

3. COMPENSATION IMPACT:
   - Employees who left earned ${left_income:,.0f} on average
   - Employees who stayed earned ${stayed_income:,.0f} on average
   - Low income bracket has highest attrition
   - Recommendation: Review compensation for underpaid roles

4. SATISFACTION DRIVERS:
   - Low job satisfaction strongly predicts attrition
   - Poor work-life balance increases turnover
   - Recommendation: Regular engagement surveys, address concerns proactively

5. KEY RISK FACTORS (from correlation analysis):
   - Short tenure (< 2 years)
   - Low satisfaction scores (< 3)
   - Lower income relative to role
   - Working overtime frequently

RECOMMENDED ACTIONS:
====================
1. Implement stay interviews for high-risk employees
2. Review compensation equity, especially for entry-level roles
3. Develop career pathing programs
4. Address work-life balance concerns (overtime policy review)
5. Quarterly engagement pulse surveys

DATA QUALITY NOTES:
===================
- Original dataset had significant quality issues (cleaned)
- Categorical inconsistencies standardized
- Numerical outliers/errors handled
- {len(df) - len(df_clean)} duplicate records removed
""")

---

# Student Exercises

Complete the following exercises to practice your EDA skills. Each exercise has a question section for you to write your code, followed by a solution section.

---

## Exercise 1: Basic Inspection (Easy)

**Tasks:**

1. How many employees are in each department? Create a bar chart.
2. What is the average tenure (years_at_company) by department?
3. Which department has the longest average tenure?

*Hint: Use `groupby()` and `value_counts()`*

In [ ]:
# Exercise 1: Your code here

# Task 1: Employees per department with bar chart


# Task 2: Average tenure by department


# Task 3: Department with longest tenure


---

## Exercise 2: Attrition Analysis (Medium)

**Tasks:**

1. Calculate the attrition rate for each job level
2. Which job level has the HIGHEST attrition rate?
3. Which job level has the LOWEST attrition rate?
4. Create a visualization comparing attrition rates across job levels

*Hint: Group by job_level and calculate the percentage where attrition == 'Yes'*

In [ ]:
# Exercise 2: Your code here

# Task 1: Attrition rate by job level


# Task 2: Highest attrition


# Task 3: Lowest attrition


# Task 4: Visualization


---

## Exercise 3: Compensation Deep-Dive (Medium)

**Tasks:**

1. Create income brackets: Low (<$4K), Medium ($4-7K), High ($7-12K), Very High (>$12K)
2. Calculate the attrition rate for each income bracket
3. Is there a clear relationship between income and attrition? Explain your finding.
4. Create a visualization showing this relationship

*Hint: Use `pd.cut()` to create income brackets*

In [ ]:
# Exercise 3: Your code here

# Task 1: Create income brackets


# Task 2: Attrition rate by bracket


# Task 3: Analysis and explanation


# Task 4: Visualization


---

## Exercise 4: Satisfaction Investigation (Medium-Hard)

**Tasks:**

1. Compare the average job_satisfaction and work_life_balance scores between:
   - Employees who LEFT
   - Employees who STAYED

2. Create side-by-side box plots comparing these metrics

3. Calculate the percentage of employees with LOW satisfaction (score 1-2) who left vs stayed

4. What conclusions can you draw about the relationship between satisfaction and attrition?

In [ ]:
# Exercise 4: Your code here

# Task 1: Compare satisfaction scores


# Task 2: Box plots


# Task 3: Low satisfaction analysis


# Task 4: Conclusions


---

## Exercise 5: Build an At-Risk Profile (Hard)

**Tasks:**

Based on your analysis, define criteria for "at-risk" employees:

1. Define at least 3 criteria that identify at-risk employees (use findings from our analysis)
   - Example criteria: Low satisfaction, short tenure, low income, etc.

2. Create a new column `is_at_risk` that flags employees meeting your criteria

3. Calculate:
   - What percentage of ALL employees are at-risk?
   - What percentage of at-risk employees actually LEFT?
   - What percentage of employees who LEFT were flagged as at-risk?

4. Create a profile summary of the "typical" at-risk employee

*Challenge: Try to maximize the percentage of leavers you can identify while minimizing false positives*

In [ ]:
# Exercise 5: Your code here

# Task 1: Define at-risk criteria


# Task 2: Create is_at_risk flag


# Task 3: Calculate percentages


# Task 4: Profile summary


---

# Congratulations!

You've completed the Employee Attrition EDA Case Study. You've learned how to:

1. **Clean messy categorical data** - standardizing inconsistent values, fixing typos, handling case/whitespace issues
2. **Clean numerical data** - handling impossible values, outliers, logical errors, and missing data
3. **Conduct systematic EDA** - following the Load → Inspect → Clean → Explore → Summarize workflow
4. **Generate business insights** - identifying attrition patterns and risk factors
5. **Create actionable recommendations** - building at-risk profiles for HR intervention

## Key Takeaways

- **Data cleaning is crucial** - Real-world data is messy; always inspect before analysis
- **Visualization reveals patterns** - Charts often show what statistics hide
- **Business context matters** - Frame findings in terms stakeholders understand
- **Actionable insights > interesting findings** - Focus on what HR can actually do

---